# Ch.4　「同じ道具」で異常を見つける ── 【講師用完全版】

| 項目 | 内容 |
|------|------|
| **使うデータ** | numpy で生成した時系列波形（200点） |
| **作るもの** | 時系列の異常スコアグラフ ＋ 異常点（スパイク）のハイライト |
| **演習時間** | 35分（問3まで完了で合格） |

> 📌 **講師メモ：** 「Ch.3 と同じ型。y に渡すものが変わるだけ」を繰り返し言葉にする。  
> ラグ特徴量の概念（過去の値を X に変換する）が最大のポイント。  
> スパイク箇所（t=50, t=150）が異常として検出されるか確認する。

---
## 🤖 Copilot の使い方

**URL：** https://copilot.microsoft.com

**質問の型（5点を揃える）：**
```
【やりたいこと】〇〇したい
【使うライブラリ】pandas / scikit-learn / numpy
【データの形】df は 200行×4列（value, lag1, lag2, lag3）の DataFrame
【環境】Python 3.8.6、Windows、JupyterLab
【困っていること】〇〇の書き方がわからない
```

---
## 準備：ライブラリとデータを読み込む

> ⚠️ **まず最初にこのセルを実行してください。**  
> 今回は CSV ではなく `numpy` でデータを人工的に生成します。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
from sklearn.linear_model import LinearRegression

print("✅ 読み込み完了")

---
## データを作成する

> ℹ️ **このセルは実行するだけでOKです。**  
> 正弦波（sin）に少しノイズを加えた時系列データを作り、  
> 2箇所だけ大きなスパイク（異常）を意図的に入れています。

In [ ]:
# 時系列データを生成する（実行するだけでOK）
np.random.seed(42)
t     = np.arange(200)
value = np.sin(t * 0.3) + np.random.normal(0, 0.2, 200)

# 2箇所にスパイク（異常）を入れる
value[50]  += 5
value[150] += 4

df = pd.DataFrame({"t": t, "value": value})
print(f"✅ データ作成完了　{len(df)} 点")

**✅ 「200 点」と表示されれば OK です。**

---
## STEP 1　データをグラフで確認する

### ❓ なぜまずグラフで確認するのか

> **「どこに異常がありそうか」を目で把握してから分析に入ります。**  
> 今回は 2箇所のスパイクが目で見えるはずです。  
> モデルがそのスパイクを「異常」として検出できるか、後で確認します。

---
### 📝 タスク 1-1　時系列データを折れ線グラフで表示してください

**何をするか：** `df["value"]` を折れ線グラフで表示する  
**使う方法：** `plt.plot(df["t"], df["value"])`

In [ ]:
# 時系列データを折れ線グラフで表示する
plt.plot(df["t"], df["value"])
plt.title("時系列データ（スパイクが2箇所あります）")
plt.xlabel("時刻 t")
plt.ylabel("値")
plt.show()

**見るポイント：**
- 大きく飛び出している点（スパイク）が 2箇所ありますか？
- それ以外の期間は一定のパターン（波）を繰り返していますか？

> 📌 「正常なパターンから大きく外れた点 ＝ 異常」がこのChapterのコアアイデアです。

---
## STEP 2　ラグ特徴量を作る

### ❓ なぜラグ特徴量が必要なのか

> **時系列データでは「過去の値」が「次の値」の予測に使えます。**  
> `shift(1)` で「1つ前の値」を新しい列として追加できます。

```
時刻  value   lag1（1つ前）  lag2（2つ前）  lag3（3つ前）
  0   0.10       NaN            NaN            NaN
  1   0.32       0.10           NaN            NaN
  2   0.55       0.32           0.10           NaN
  3   0.71       0.55           0.32           0.10   ← X に使う
  …
```

> X = [lag1, lag2, lag3]（過去3点の値）  
> y = value（現在の値）  
> → Ch.3 と全く同じ `fit → predict` の型が使えます！

---
### 📝 タスク 2-1　lag1・lag2・lag3 を作ってください

**何をするか：** `df["value"].shift(N)` で N 個前の値を新しい列にする

In [ ]:
# 1つ前・2つ前・3つ前の値をラグ特徴量として追加する
df["lag1"] = df["value"].shift(1)
df["lag2"] = df["value"].shift(2)
df["lag3"] = df["value"].shift(3)

---
### 📝 タスク 2-2　先頭の欠損行（NaN）を削除してください

**何をするか：** `shift()` で生じた先頭の NaN 行を `dropna()` で削除する

In [ ]:
# NaN のある行を削除する（shift で生じた先頭3行）
df = df.dropna()
print(f"削除後: {len(df)} 行")

---
### 📝 タスク 2-3　先頭5行を表示して確認してください

In [ ]:
# 先頭5行を確認する（lag1〜3 が追加されているか）
df.head()

**✅ チェックポイント**
- [ ] `lag1` `lag2` `lag3` の列が追加されているか
- [ ] 行数が 197 になっているか（200 - 3 = 197）
- [ ] 先頭の NaN がなくなっているか

---
## 問1　正常期間だけでモデルを学習する ★☆☆

### ❓ なぜ「正常期間だけ」で学習するのか

> **「正常なパターン」を覚えさせるためです。**  
> 異常を含むデータで学習すると「異常も正常」と覚えてしまいます。  
> 今回はスパイクが少ない最初の 80点を「正常期間」として使います。

---
### 📝 タスク 1-1　X（特徴量）と y（正解）を作ってください

**何をするか：** lag1〜3 を X に、value を y にする

In [ ]:
# X（ラグ特徴量）と y（現在の値）を作る
X = df[["lag1", "lag2", "lag3"]]
y = df["value"]

---
### 📝 タスク 1-2　最初の 80点を「正常期間」として取り出してください

**何をするか：** `.iloc[:80]` で最初の 80行を取り出す

In [ ]:
# 最初の 80点を訓練データ（正常期間）として使う
X_train = X.iloc[:80]
y_train = y.iloc[:80]
print(f"訓練データ: {len(X_train)} 点")

---
### 📝 タスク 1-3　LinearRegression でモデルを学習させてください

**何をするか：** Ch.3 と全く同じ `model.fit()` の書き方  
**違いはここだけ：** `RandomForestClassifier` → `LinearRegression`

In [ ]:
# LinearRegression モデルを作って学習する（Ch.3 と同じ型！）
model = LinearRegression()
model.fit(X_train, y_train)
print("✅ 学習完了")

**✅ チェックポイント**
- [ ] X に lag1・lag2・lag3 の3列が入っているか
- [ ] `model.fit()` がエラーなく完了したか

---
## 問2　全期間を予測して異常スコアを計算する ★★☆

### ❓ なぜ「全期間」で予測するのか

> **正常パターンを覚えたモデルに「全期間」を予測させると、**  
> 正常な箇所は予測が当たり、異常な箇所だけ予測が大きく外れます。  
> その「外れ幅（予測誤差）」が異常スコアになります。

```
予測値  ≒ 観測値  → 予測誤差 ≒ 0  → 正常
予測値 ≠≠ 観測値  → 予測誤差 大   → 異常！
```

---
### 📝 タスク 2-1　全期間（197点）を予測してください

**何をするか：** Ch.3 と同じ `model.predict()` を全データに対して実行する

In [ ]:
# 全期間を予測する（Ch.3 と同じ型！）
y_pred = model.predict(X)

---
### 📝 タスク 2-2　異常スコア（予測誤差）を計算してください

**何をするか：** `|観測値 − 予測値|` を計算する  
**`abs()` とは：** 絶対値（マイナスをプラスにする）

In [ ]:
# 異常スコア = |観測値 - 予測値|
score = abs(df["value"].values - y_pred)
print(f"スコアの最大値: {score.max():.2f}")
print(f"スコアの平均値: {score.mean():.2f}")

---
### 📝 タスク 2-3　異常スコアをグラフで確認してください

**何をするか：** `score` を折れ線グラフで表示して、スパイク箇所が高くなっているか確認する

In [ ]:
# 異常スコアをグラフで表示する
plt.plot(df["t"], score)
plt.title("異常スコア（予測誤差）")
plt.xlabel("時刻 t")
plt.ylabel("異常スコア")
plt.show()

**✅ チェックポイント**
- [ ] 異常スコアのグラフが表示されたか
- [ ] t=50 と t=150 付近でスコアが大きく跳ね上がっているか
- [ ] それ以外の箇所はスコアが低く（0〜0.5程度）安定しているか

---
## 問3　しきい値で異常を判定してグラフにハイライトする ★★☆（最重要）

### ❓ なぜしきい値を使うのか

> **「どのくらい外れたら異常か」の基準を決めるためです。**  
> `np.percentile(score, 95)` は「上位5%のスコア」を基準にします。  
> この基準を超えた点を「異常」と判定します。

```
異常スコアの分布:
 ──────────────┬──── ← np.percentile(score, 95)
  ここは正常  │ ここは異常（上位5%）
```

---
### 📝 タスク 3-1　しきい値を設定してください

**何をするか：** `np.percentile(score, 95)` でスコアの上位 5% を基準にする

In [ ]:
# しきい値：スコアの上位 5% を異常とみなす
threshold = np.percentile(score, 95)
print(f"しきい値: {threshold:.2f}")

---
### 📝 タスク 3-2　異常フラグを作ってください

**何をするか：** スコアがしきい値を超えているかを True/False で判定する

In [ ]:
# しきい値を超えたら異常（True）
is_anomaly = score > threshold
print(f"異常と判定した点: {is_anomaly.sum()} 点")

---
### 📝 タスク 3-3　正常と異常を色分けしてグラフを描いてください

**何をするか：** 正常は青い折れ線、異常点だけ赤い点（●）で上書きする  
**Copilot に聞く場合：** 「折れ線グラフと散布図を重ねて描く方法を教えて」

In [ ]:
# 正常：青い折れ線、異常：赤い点でハイライト
plt.plot(df["t"], df["value"], color="steelblue", label="正常")
plt.scatter(df["t"][is_anomaly], df["value"][is_anomaly],
            color="red", zorder=5, label="異常")
plt.title("異常検知の結果")
plt.xlabel("時刻 t")
plt.legend()
plt.show()

---
### 📝 タスク 3-4　結果から気づいたことを書いてください（AI 禁止）

> ⛔ **この考察は自分で書いてください。AI は使わないこと。**

> 📌 **なぜ大事なのか：**  
> 「しきい値を何%にするか」はデータではなくビジネスが決めます。  
> 見逃し（異常を見落とす）と誤検知（正常を異常と判断する）のバランスを  
> ビジネスの文脈で判断するのがデータサイエンティストの仕事です。

---

**【発見】赤い点（異常と判定された箇所）は t=50 と t=150 付近にありましたか？**

>

**【解釈】スパイク以外に誤検知はありましたか？あった場合、どう対処できますか？**

>

**【Ch.3 との接続】Ch.3 と同じコードの型（fit → predict）を使いましたが、何が違いましたか？**

>

**✅ チェックポイント**
- [ ] グラフに赤い点が表示されているか
- [ ] 赤い点が t=50 と t=150 付近に集中しているか
- [ ] 考察を自分の言葉で書けたか

---
## 問4（発展）　しきい値のパーセンタイルを変えて観察する ★★★

> ⏰ **時間が余った人だけ取り組んでください**

### ❓ しきい値はどう決めるか

> **「何%を異常とみなすか」はビジネス判断です。**

| パーセンタイル | 意味 | 検出件数 |
|--------------|------|---------|
| 90% | 上位10%を異常 | 多め（誤検知が増えやすい） |
| 95% | 上位 5%を異常 | バランス型（今回の設定） |
| 99% | 上位 1%を異常 | 少なめ（見逃しが増えやすい） |

---
### 📝 タスク 4-1　パーセンタイルを 90・95・99 に変えて検出件数を比べてください

**何をするか：** `np.percentile(score, N)` の N を変えて `is_anomaly.sum()` を確認する

In [ ]:
# パーセンタイルを変えて検出件数を比べる
for p in [90, 95, 99]:
    th = np.percentile(score, p)
    n  = (score > th).sum()
    print(f"上位{100-p:2d}%（p={p}）: しきい値={th:.2f}  検出={n}点")

**✅ チェックポイント**
- [ ] パーセンタイルが高い（99%）ほど検出件数が少なくなっているか
- [ ] パーセンタイルが低い（90%）と誤検知が増えるか
- [ ] ビジネス上どのパーセンタイルが適切か、理由を考えてみてください